# Jour 3 — Interpréter les représentations d’un CNN

> **Version corrigée** — les cellules de solution sont signalées.


## Objectifs

- comprendre champ réceptif, activation et feature map
- extraire les sorties de couches intermédiaires
- construire une carte d’occlusion
- distinguer interprétation utile et preuve de causalité


## Reprendre l’architecture d’un CNN

![Pipeline général d’un CNN](../../ressources/illustrations/jour_02/03_architecture_cnn.png)

Un CNN transforme progressivement une image. Les convolutions cherchent des motifs locaux, le pooling réduit la taille spatiale et les dernières couches produisent une décision.

Le mot **activation** désigne la sortie numérique d’une couche pour une entrée précise. Les poids d’un filtre sont appris et restent les mêmes pour toutes les images ; les activations changent selon l’image fournie.

Observer les couches intermédiaires ne traduit pas automatiquement le raisonnement du réseau en langage humain. Cela permet néanmoins de vérifier où et comment certaines représentations réagissent.


## Le champ réceptif

![Croissance du champ réceptif](../../ressources/illustrations/jour_03/01_champ_receptif.png)

Le **champ réceptif** d’une activation est la zone de l’image d’entrée susceptible de l’influencer.

Une convolution 3 × 3 regarde localement une zone de 3 × 3. Les couches suivantes combinent des activations déjà construites à partir de plusieurs pixels. Leur champ réceptif devient donc plus large.

Le champ réceptif théorique indique la zone possible d’influence. Tous les pixels de cette zone n’ont pas forcément la même importance pour une prédiction donnée.


### Stride, padding et pooling modifient ce que l’on observe

Le **stride**, ou pas, indique de combien de pixels le filtre se déplace. Un stride de 1 examine toutes les positions voisines ; un stride de 2 réduit la résolution de la sortie.

Le **padding** ajoute une bordure technique. Avec `padding="same"` et un stride de 1, la largeur et la hauteur sont conservées. Sans padding, les bords sont moins visités et la carte rétrécit.

Le **pooling** résume une petite zone. Il augmente indirectement le contexte couvert par les activations suivantes, mais perd une partie de la précision spatiale.

Lorsque l’on compare deux feature maps, il faut donc vérifier leur couche, leur forme et les transformations précédentes. Une carte 4 × 4 ne peut pas localiser un détail avec la même précision qu’une carte 8 × 8.


## Une feature map par filtre

![Production de plusieurs feature maps](../../ressources/illustrations/jour_03/01_feature_maps.png)

`Conv2D(16, ...)` apprend 16 filtres et produit 16 feature maps. Chaque carte conserve une organisation spatiale : une valeur élevée indique une réponse forte du filtre à cet endroit.

Un filtre n’est pas programmé comme « détecteur de bord vertical ». Il reçoit cette utilité parce que ses poids évoluent pendant l’entraînement. Certains motifs deviennent visuellement interprétables ; d’autres restent des combinaisons difficiles à nommer.

La visualisation dépend aussi de l’échelle choisie. Une carte presque constante peut sembler contrastée si chaque image est automatiquement remise entre son minimum et son maximum.


## Entraîner un petit CNN nommé couche par couche

Nous réutilisons `digits`. Les noms `conv_1` et `conv_2` permettront de récupérer les activations sans dépendre de la position des couches dans la liste Keras.

Le test reste séparé. Il sert ici à sélectionner un exemple à expliquer après l’entraînement, pas à corriger les poids.

Nous ajoutons ici une seconde convolution au CNN simple du jour 2, pour observer comment elle combine les réponses des premiers filtres.


Nous reprenons le chargement et la normalisation du jour 2. `np.newaxis` ajoute le canal de gris. La première convolution produit `(8, 8, 16)` ; après le pooling, nous ajoutons une seconde convolution qui produit `(4, 4, 32)`. Elle reçoit les réponses des premiers filtres, pas directement les pixels.

Ici, `validation_split=0.20` réserve la dernière fraction du tableau de train pour mesurer la validation. Ces images ne participent pas aux corrections de poids. Le test est gardé à part comme précédemment.


In [ ]:
# Réglages de l’environnement, avant le chargement de TensorFlow.
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")

# NumPy gère les tableaux et les calculs numériques.
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

# Fixe le hasard pour faciliter les comparaisons.
tf.keras.utils.set_random_seed(42)
try:
    # Limite les ressources utilisées par cette petite démonstration.
    tf.config.threading.set_intra_op_parallelism_threads(1)
    tf.config.threading.set_inter_op_parallelism_threads(1)
except RuntimeError:
    pass

# Charge les images et leurs labels ensemble.
digits = load_digits()
# Convertit les pixels en décimaux et les ramène à l’échelle [0, 1].
images = digits.images.astype("float32") / 16.0


In [ ]:
# Conserve un chiffre entier comme bonne réponse par image.
labels = digits.target.astype("int64")
# Sépare ensemble les entrées et les réponses pour garder leur association.
X_train, X_test, y_train, y_test = train_test_split(
    images, labels, test_size=0.25, stratify=labels, random_state=42
)
# Ajoute le canal de gris attendu par les convolutions.
X_train_cnn = X_train[..., np.newaxis]
# Applique le même format au test.
X_test_cnn = X_test[..., np.newaxis]

# Les couches sont reliées dans l’ordre de cette liste.
cnn = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(8, 8, 1)),
    tf.keras.layers.Conv2D(16, 3, padding="same", activation="relu", name="conv_1"),
    tf.keras.layers.MaxPooling2D(2, name="pool_1"),
    # Combine les premières cartes avec 32 filtres supplémentaires.
    tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu", name="conv_2"),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(10, activation="softmax"),
])
# Associe le réseau, sa loss et l’optimiseur ; aucun poids ne change encore.
cnn.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
# Entraîne les poids sur le train et observe la validation séparément.
cnn.fit(X_train_cnn, y_train, validation_split=0.20, epochs=10, batch_size=32, verbose=0)


In [ ]:
# Mesure les nouvelles images sans corriger les poids.
print("Accuracy test :", round(cnn.evaluate(X_test_cnn, y_test, verbose=0)[1], 3))


## Créer un modèle d’observation

Un modèle Keras peut partager les mêmes couches qu’un autre modèle tout en exposant des sorties différentes.

L’`inspecteur` ci-dessous reçoit la même image que le CNN, mais renvoie les activations de `conv_1` et `conv_2`. Il n’est pas réentraîné : il donne accès aux calculs déjà appris.

La première sortie possède une taille 8 × 8 × 16. Après le pooling, la seconde convolution produit 4 × 4 × 32.


`tf.keras.Model(inputs=..., outputs=...)` crée une vue du réseau qui s'arrête aux sorties choisies. Une liste de deux sorties renvoie deux tableaux.

La tranche `sample_index:sample_index+1` garde un batch d'une image. Écrire seulement `[sample_index]` retirerait cet axe. Dans `activation_1[0, :, :, i]`, on prend la première image, toutes les lignes et colonnes, puis la carte `i`. La forme annoncée permet de contrôler ce découpage avant de regarder les couleurs.


In [ ]:
# Crée une vue des activations apprises, sans nouveau réseau à entraîner.
inspecteur = tf.keras.Model(
    inputs=cnn.inputs,
    outputs=[
        cnn.get_layer("conv_1").output,
        cnn.get_layer("conv_2").output,
    ],
)

sample_index = 0
# Garde l’axe du batch pour une seule image et récupère les deux sorties.
activation_1, activation_2 = inspecteur.predict(X_test_cnn[sample_index:sample_index+1], verbose=0)
print("conv_1 :", activation_1.shape)
print("conv_2 :", activation_2.shape)

# Réserve les emplacements graphiques, sans modifier les données.
fig, axes = plt.subplots(2, 6, figsize=(11, 4))
for i in range(6):
    axes[0, i].imshow(activation_1[0, :, :, i], cmap="viridis")
    axes[0, i].set_title(f"conv1 #{i}")
    axes[1, i].imshow(activation_2[0, :, :, i], cmap="magma")
    axes[1, i].set_title(f"conv2 #{i}")
    axes[0, i].axis("off")
    axes[1, i].axis("off")
plt.tight_layout()


In [ ]:
# Affiche le résultat des calculs précédents.
plt.show()


## Mesurer une sensibilité par occlusion

![Principe de la carte d’occlusion](../../ressources/illustrations/jour_03/01_occlusion.png)

L’**occlusion** masque une petite zone puis recalcule le score de la classe étudiée. Si le score baisse fortement, cette zone contribuait à la décision actuelle.

Nous répétons l’opération pour chaque pixel. Les 64 images modifiées sont regroupées dans un batch afin d’éviter 64 appels séparés au modèle.

La valeur calculée est une différence de score. Une valeur positive indique que masquer la zone réduit le score ; une valeur négative signifie que le masque l’augmente.


Une **fonction Python**, introduite par `def`, regroupe des calculs réutilisables ; `return` rend leur résultat. Ici, `occlusion_map` fabrique des copies de l'image, masque un pixel différent dans chacune, puis mesure la baisse du score.

`np.repeat` crée ces copies ; `image[np.newaxis, ...]` ajoute l'axe du batch. `divmod(position, width)` convertit un numéro de case en numéro de ligne et de colonne. `reshape(height, width)` range la liste de résultats dans la grille originale, sans changer les nombres.

`argmax(axis=1)` donne la classe préférée de chaque image ; `np.flatnonzero` retrouve les positions où une condition est vraie. Nous choisissons un exemple correctement classé. Une couleur positive signale ici qu'effacer ce pixel fait baisser le score de la vraie classe ; cela mesure une sensibilité, pas une preuve de causalité.


In [ ]:
# Fonction réutilisable : mesure l’effet d’un pixel effacé sur le score choisi.
def occlusion_map(model, image, class_index):
    height, width, _ = image.shape
    # Copie l’image une fois par position à masquer.
    variants = np.repeat(image[np.newaxis, ...], height * width, axis=0)
    for position in range(height * width):
        # Convertit une position linéaire en ligne et colonne.
        row, col = divmod(position, width)
        # Efface un pixel différent dans chaque copie.
        variants[position, row, col, 0] = 0.0

    # Score de référence, avec l’image intacte.
    original_score = model.predict(image[np.newaxis, ...], verbose=0)[0, class_index]
    # Score de la même classe après chaque masquage.
    masked_scores = model.predict(variants, verbose=0)[:, class_index]
    # Calcule la baisse du score et la remet en grille.
    return original_score - masked_scores.reshape(height, width)

# Une ligne de scores par image.
probabilities = cnn.predict(X_test_cnn, verbose=0)
# Position du score le plus élevé de chaque ligne.
predictions = probabilities.argmax(axis=1)
# Position du score le plus élevé de chaque ligne.
correct_indices = np.flatnonzero(predictions == y_test)
explained_index = int(correct_indices[0])


In [ ]:
# Applique le test d’occlusion à l’image choisie.
importance = occlusion_map(cnn, X_test_cnn[explained_index], int(y_test[explained_index]))

# Réserve les emplacements graphiques, sans modifier les données.
fig, axes = plt.subplots(1, 2, figsize=(7, 3))
axes[0].imshow(X_test[explained_index], cmap="gray_r")
axes[0].set_title(f"Chiffre {y_test[explained_index]}")
axes[1].imshow(importance, cmap="coolwarm")
axes[1].set_title("Baisse du score si masqué")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
# Affiche le résultat des calculs précédents.
plt.show()


## Ce que la carte permet — et ne permet pas — de dire

La carte décrit la **sensibilité locale** d’une prédiction à une modification artificielle. Elle aide à voir si le réseau s’appuie sur le tracé du chiffre ou sur une zone inattendue.

Elle ne prouve pas qu’une zone est une cause au sens scientifique. Le remplacement par zéro peut lui-même créer un motif inhabituel. La résolution 8 × 8 rend aussi la carte très grossière.

Une interprétation utile combine plusieurs éléments : exemple original, classe réelle, classe prédite, confiance, activations et comportement sous modification.


## À vous de jouer — expliquer un autre exemple

Sélectionnez le deuxième exemple correctement classé, calculez sa carte d’occlusion avec `occlusion_map`, puis affichez l’image et la carte côte à côte.


In [ ]:
# Correction
second_index = int(correct_indices[1])
# Applique le test d’occlusion à l’image choisie.
second_importance = occlusion_map(
    cnn,
    X_test_cnn[second_index],
    int(y_test[second_index]),
)

# Réserve les emplacements graphiques, sans modifier les données.
fig, axes = plt.subplots(1, 2, figsize=(7, 3))
axes[0].imshow(X_test[second_index], cmap="gray_r")
axes[0].set_title(f"Réel {y_test[second_index]} — prédit {predictions[second_index]}")
axes[1].imshow(second_importance, cmap="coolwarm")
axes[1].set_title("Carte d’occlusion")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
# Affiche le résultat des calculs précédents.
plt.show()


## À retenir

Une feature map montre la réponse d’un filtre à différents endroits. Une carte d’occlusion mesure comment le score varie lorsqu’une zone est masquée. Ces outils rendent le réseau plus inspectable, sans transformer ses calculs en explication certaine.
